# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rupamsingh0907-beep/Flyintern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane:** Lane 2, Refresh / Content Opportunity Scoring (provisional).

**Task type:** ranking / scoring. Each page gets a priority score, and the pages are ordered by it, so the output is a ranked list. The question this answers is "which pages should a reviewer look at first?"

**Why not the other types:**
- *Classification* would only label each page yes/no (for example, declining or not). A yes/no label does not say which of thousands of flagged pages to open first, and the reviewer can only check a limited number.
- *Clustering* would find groups of similar pages without saying which group needs attention first.

**Action the output supports:** a content reviewer opens the top-ranked pages and chooses one action for each: refresh, expand, protect, prune or monitor.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy target:** `declining_proxy`, which is 1 when a page's `trend_direction` is "down" and 0 otherwise.

This is a proxy, not an observed future outcome. `trend_direction` comes from the current time window, so it says a page is *currently trending down*, not that it *will* decline. A stronger target would label decline in a later window than the one the inputs come from.

**Leakage note:** `trend_direction` and `trend_pct` define the target, so I keep them out of the input columns. If they were inputs, a model would only learn my labeling rule instead of finding real signals.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric:** precision@K with K = 50, my assumed reviewer capacity. It is the share of the top 50 ranked pages that are actually trending down.

**Chosen before any modeling, with two reference points:**
- Random baseline: about 54.2% of pages are trending down, so a random top 50 would hit roughly 27. A useful ranking has to beat this clearly.
- Rule baseline: a transparent rule-based flag (for example, stale and visible pages). The ranking has to beat this too to be worth the extra complexity.

Because the target is a proxy, I treat any result as directional.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one page** (one `content_id`), with its visibility, engagement and freshness signals. These are the inputs a scoring model would use.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")


In [4]:
unit_cols = [
    "content_id", "client_id",
    "impressions_90d", "sessions_90d", "ctr", "avg_position",
    "engagement_rate", "content_age_days", "days_since_last_update",
]

pages = df[unit_cols].copy()
pages.head(10)

,content_id,client_id,impressions_90d,sessions_90d,ctr,avg_position,engagement_rate,content_age_days,days_since_last_update
0,content_304f48230142,client_f369cb89fc,3803,17,0.76,10.6,5.88,187,20
1,content_a1fb4e703a9e,client_4e07408562,15320,9,0.05,20.3,0.00,445,25
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,0.00,141,20
3,content_331d6c4de07b,client_19581e27de,11751,78,0.49,6.2,1.28,463,22
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,0.13,44.0,0.00,263,14
5,content_d4084a4bc775,client_f369cb89fc,3970,5,0.03,8.5,0.00,147,20
6,content_9a34b442b552,client_8722616204,20,1,0.00,7.0,0.00,90,20
7,content_a63219c6e95a,client_19581e27de,1724,28,0.06,21.2,3.57,445,22
8,content_5e6c160719bc,client_6208ef0f77,32574,68,0.09,46.0,5.88,90,20
9,content_c27558df2b0c,client_19581e27de,1240,3,0.16,4.9,0.00,257,104


In [5]:
pages["declining_proxy"] = (df["trend_direction"] == "down").astype(int)

print(pages[["content_id", "declining_proxy"]].head(10))
print(pages["declining_proxy"].value_counts())
print(pages["declining_proxy"].mean().round(3))

             content_id  declining_proxy
0  content_304f48230142                1
1  content_a1fb4e703a9e                1
2  content_9aa793d4d895                1
3  content_331d6c4de07b                0
4  content_d99b7a2d90ca                1
5  content_d4084a4bc775                1
6  content_9a34b442b552                1
7  content_a63219c6e95a                0
8  content_5e6c160719bc                1
9  content_c27558df2b0c                1
declining_proxy
1    16262
0    13738
Name: count, dtype: int64
0.542


This sketch of the target column shows about 54% of pages labeled 1 (currently trending down), matching my Week 1 number. It is a current-window proxy, so it should be read as "currently trending down," not "will decline."

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
rule_flag = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
print("pages flagged by the rule:", rule_flag.sum())

pages flagged by the rule: 17


A fixed rule such as "stale and visible" (not updated in 180+ days and 500+ impressions) flags **[X] pages** with one cutoff and treats them all the same. A reviewer who can check about 50 pages cannot use an unordered list that long. A scoring approach could combine several signals (visibility, freshness, engagement, position, recent movement) into an ordering, so the top of the queue holds the pages with the strongest evidence.

This is an argument, not yet a result. In later weeks the ranking has to beat the rule on precision@50 to justify using ML. The lane guide's starter run suggests it might (0.240 for the rules, 0.740 for a random forest), but that comes from a 30,000-row slice, so it is directional only.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.